# T3-bonus · Workflows as code

## Goal

Pack the workflow YAML into the solution alongside the agent, and promote
it dev to test with environment-variable rebinding — the finance MCP
endpoint in test is not the finance MCP endpoint in dev, and the workflow
must not hardcode either.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
assert (Path("../agents/contract-renewal-desk/workflows/renewal-check.yaml")).exists(), "run 09-11 first"


## Concept

`renewal-check.yaml` currently points at `finance-ops-mcp` by name, which
resolves per-environment via a connection reference — that's already
environment-agnostic. What isn't yet is anything a notebook hardcoded
along the way (spend thresholds, review dollar amounts) — those move to
environment variables now, defined once in the solution and given
different values per environment, so `test`'s $2M human-review threshold
doesn't have to match `prod`'s.


## Build


In [ ]:
import yaml
from pathlib import Path
workflow_path = Path("../agents/contract-renewal-desk/workflows/renewal-check.yaml")
workflow = yaml.safe_load(workflow_path.read_text())

# Replace the hardcoded threshold with an environment variable reference
for step in workflow["steps"]:
    if step.get("id") == "checkHighValue":
        step["condition"] = step["condition"].replace("2000000", "@{env.humanReviewThresholdUsd}")
workflow_path.write_text(yaml.dump(workflow, sort_keys=False))


In [ ]:
import yaml
from pathlib import Path
env_vars_path = Path("../agents/contract-renewal-desk/environment-variables.yaml")
env_vars_path.write_text(yaml.dump({
    "humanReviewThresholdUsd": {"type": "number", "defaultValue": 2000000},
}, sort_keys=False))


In [ ]:
from csx.pac import copilot_pack
from pathlib import Path
copilot_pack(Path("../agents/contract-renewal-desk"), Path("../dist/crd.zip"))


## Verify

Same harness, same golden set, every notebook.


In [ ]:
import subprocess
# Import into test with a different threshold, prove the rebind takes effect
result = subprocess.run([
    "pac", "solution", "import", "--path", "../dist/crd.zip",
    "--environment", "$TEST_ENVIRONMENT_URL",
    "--settings-file", "test-env-overrides.json",  # {"humanReviewThresholdUsd": 500000}
], capture_output=True, text=True)
print(result.returncode)


## Cost


In [ ]:
print("Packing is free (no auth, no environment touched). Import + one verification invocation in test meters modestly.")


## Teardown


In [ ]:
print("No teardown — environment-variable rebinding is now the standing pattern for dev/test/prod promotion (finished in 25).")
